# Person 3 — Hugging Face API cost comparison

The workload is Person 2's inference-tier contract,
`hestia-a7-256in-500out-r1-inference-only-v1`:

- 1 request per daily active user
- 256 input tokens and 500 output tokens
- six traffic levels from 1,000 to 100,000,000 daily users
- 15 registry models, 90 rows

Token price is the Hugging Face Inference Providers listing observed on
23 Oct 2026 for `meta-llama/Llama-3.1-8B-Instruct` through Novita:
$0.02 per million input tokens and $0.05 per million output tokens.
The local roster ids are not all on that catalog, so every row uses this
one comparable hosted model.

The API gateway is a 2-vCPU Runpod CPU pod at the catalog example rate of
$0.04 per vCPU-hour, plus 20 GB of container disk at $0.10 per GB-month.
A planning rate of 50 requests per second per gateway is an assumption,
sized with Person 2's utilization, peak, and service-time factors.
Django, the frontend, and the database are excluded on both sides.

`python -m a7.person3.build_api_costs` rewrites `a7/results/api_costs.csv`
and checks that all 90 rows join to `a7/results/local_costs.csv`.


In [1]:
import csv
rows = list(csv.DictReader(open("a7/results/api_costs.csv")))
print("rows", len(rows))
print("models", len({r["model_id"] for r in rows}))
print("traffic", len({r["traffic_category"] for r in rows}))
cols = ["traffic_category","daily_active_users","total_daily_token_load","estimated_api_inference_cost_usd","machines_required_api_infrastructure","total_daily_cost_usd"]
for row in rows:
    if row["model_id"] == "Qwen/Qwen2.5-7B-Instruct":
        print(" | ".join(row[c] for c in cols))


rows 90
models 15
traffic 6
Prototype | 1000 | 500000 | 0.03012 | 1 | 2.016787
Early Startup | 10000 | 5000000 | 0.3012 | 1 | 2.287867
Growing Product | 100000 | 50000000 | 3.012 | 1 | 4.998667
Large Platform | 1000000 | 500000000 | 30.12 | 1 | 32.106667
Mass Consumer App | 10000000 | 5000000000 | 301.2 | 10 | 321.066667
Global Platform | 100000000 | 50000000000 | 3012.0 | 97 | 3204.706667


## Why one paid model appears in all 90 rows

Each local model is a possible way to provide the same household instruction service.
The API scenario holds the external alternative fixed at Llama-3.1-8B-Instruct via
Novita/Hugging Face, while changing the local model and traffic. This avoids mixing
provider-rate changes into the local-capacity comparison. All 15 API alternatives
therefore have the same cost at a given traffic level.

This is a common-service price scenario, not 15 exact-model API offers or a
parameter-matched comparison. The hosted candidate has not been tested on the
shared prompts, so equivalent output quality is not established. Closer size-matched
providers or measured quality-qualified API alternatives would be a useful extension;
the cost result alone does not justify deploying an unqualified route. Rates are the
supplied October 23 assumptions, not a newly verified live deployment quote.

## Comparison with local hosting

All 90 API rows join to Person 2's local table on model, traffic, request
count, token counts, and assumptions id.

Under this rate card the API total is lower for every model at every
sampled traffic level. At 1,000 users the API day is about $2.02, against
about $12.33 for the smallest local GPU plan and $37 for the 7B plans.
At 100 million users the same API day is about $3,205, while the local
plans are hundreds of thousands to millions of dollars because replica
count grows with token load.

No sampled scale makes local hosting cheaper. The published Llama 8B token
price stays below Person 2's break-even threshold even after the CPU
gateway is included. Local hosting would become the cheaper inference tier
only if the blended API rate rose above that threshold, or if the deployment
needed the model to stay on machines the team controls.
